In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import RMSE, SMAPE

In [2]:
Y_COL    = 'Sum of кВт'
ID_COL   = 'EIC-код_cat'
DS_COL   = 'datetime'

WEATHER_COLS = [
    'temperature_2m', 'apparent_temperature', 'dew_point_2m',
    'relative_humidity_2m', 'precipitation', 'rain', 'snowfall',
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'surface_pressure', 'wind_speed_10m', 'wind_direction_10m',
    'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation',
    'direct_normal_irradiance',
]
TIME_COLS  = ['month', 'hour', 'dow', 'season']   # numeric versions, created in prepare_nf
FUTR_EXOG  = WEATHER_COLS + TIME_COLS
STAT_EXOG  = ['lat', 'lon']                        # per-location static features

HORIZON    = 744    # ~1 month of hourly steps  (used for val/test evaluation)
INPUT_SIZE = 3 * HORIZON  # 3 months of context (was 2)

In [3]:
def smallest_int_dtype(min_val, max_val, signed=True):
    if signed:
        for dtype in ['int8', 'int16', 'int32', 'int64']:
            info = np.iinfo(dtype)
            if info.min <= min_val <= max_val <= info.max:
                return dtype
    else:
        for dtype in ['uint8', 'uint16', 'uint32', 'uint64']:
            info = np.iinfo(dtype)
            if 0 <= min_val <= max_val <= info.max:
                return dtype
    return 'int64'


def optimize_df_for_memory(df):
    meta = {}
    for col in df.columns:
        s = df[col]
        unique_non_null = set(s.dropna().unique())
        if unique_non_null.issubset({0, 1, True, False}) and col == 'Група':
            df[col] = s.astype('bool')
            meta[col] = {'stored_as': 'bool', 'scale': 1}
            continue
        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dtype)
            meta[col] = {'stored_as': dtype, 'scale': 1}
            continue
        if pd.api.types.is_float_dtype(s):
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype('float32')
                meta[col] = {'stored_as': 'float32', 'scale': 1}
                continue
            decimals = non_null.astype(str).apply(
                lambda x: len(x.split('.')[1].rstrip('0')) if '.' in x else 0
            ).max()
            if decimals <= 3:
                scale  = 10 ** decimals
                scaled = np.round(s * scale)
                mn, mx = int(np.nanmin(scaled)), int(np.nanmax(scaled))
                int_dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
                if np.dtype(int_dtype).itemsize < np.dtype('float32').itemsize:
                    df[col] = scaled.astype(int_dtype)
                    meta[col] = {'stored_as': int_dtype, 'scale': scale}
                    continue
            df[col] = s.astype('float32')
            meta[col] = {'stored_as': 'float32', 'scale': 1}
    return df, meta


def add_cat_helpers(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['Month_cat']       = df['datetime'].dt.month.astype(str)
    df['Day_cat']         = df['datetime'].dt.day.astype(str)
    df['Hour_cat']        = df['datetime'].dt.hour.astype(str)
    df['day_of_week_cat'] = df['datetime'].dt.dayofweek.astype(str)
    season_map = {12: 'winter', 1: 'winter', 2: 'winter',
                   3: 'spring', 4: 'spring', 5: 'spring',
                   6: 'summer', 7: 'summer', 8: 'summer',
                   9: 'autumn', 10: 'autumn', 11: 'autumn'}
    df['season_cat'] = df['datetime'].dt.month.map(season_map)
    return df


def load_and_prepare(path):
    df = pd.read_parquet(path).reset_index(drop=True)
    df.columns = df.columns.str.replace('.', '_', regex=False)
    df, _ = optimize_df_for_memory(df)
    df = add_cat_helpers(df)
    for col in df.columns:
        if col.endswith('_cat'):
            df[col] = df[col].astype(str)
    try:
        df[Y_COL] = df[Y_COL].astype('float32')
    except Exception:
        print(f'No Y_col: {Y_COL}')
    df = df.sort_values([ID_COL, DS_COL]).reset_index(drop=True)
    try:
        df.drop(columns=['Ціна розподілу ЕЕ', 'Ціна ЕЕ', 'Money_spent'], inplace=True)
    except Exception:
        pass
    return df


def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return float(np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

In [4]:
SEASON_MAP = {'winter': 0, 'spring': 1, 'summer': 2, 'autumn': 3}


def prepare_nf(df, has_y=True):
    """Convert a raw DataFrame to neuralforecast format."""
    df = df.copy()

    # Normalise datetime: strip timezone if present
    ds = pd.to_datetime(df[DS_COL])
    if ds.dt.tz is not None:
        ds = ds.dt.tz_convert(None)
    df['ds'] = ds

    df = df.rename(columns={ID_COL: 'unique_id'})
    if has_y:
        df = df.rename(columns={Y_COL: 'y'})

    # Numeric time features (neuralforecast requires numeric exog)
    df['month']  = df['Month_cat'].astype(int)
    df['hour']   = df['Hour_cat'].astype(int)
    df['dow']    = df['day_of_week_cat'].astype(int)
    df['season'] = df['season_cat'].map(SEASON_MAP).astype(int)

    # Ensure all weather cols are float32
    for col in WEATHER_COLS:
        df[col] = df[col].astype('float32')

    keep = ['unique_id', 'ds'] + (['y'] if has_y else []) + FUTR_EXOG
    return df[keep].sort_values(['unique_id', 'ds']).reset_index(drop=True)

In [5]:
train = load_and_prepare('data/silver_money_calc/train.parquet')
val   = load_and_prepare('data/silver_money_calc/val.parquet')
test  = load_and_prepare('data/silver_money_calc/test.parquet')

train_nf = prepare_nf(train)
val_nf   = prepare_nf(val)
test_nf  = prepare_nf(test)

print('train:', train_nf.shape, ' val:', val_nf.shape, ' test:', test_nf.shape)

train: (4864324, 25)  val: (304499, 25)  test: (304152, 25)


In [6]:
# One row per unique_id with static numeric features
static_df = (
    train[[ID_COL, 'Широта', 'Довгота']]
    .drop_duplicates(ID_COL)
    .rename(columns={ID_COL: 'unique_id', 'Широта': 'lat', 'Довгота': 'lon'})
    .reset_index(drop=True)
)
static_df[['lat', 'lon']] = static_df[['lat', 'lon']].astype('float32')
print(static_df.shape)
static_df.head()

(410, 3)


,unique_id,lat,lon
0,62Z0008583037334,48.442429,22.192190
1,62Z0011230718431,51.542107,31.262997
2,62Z0096677872985,48.569660,22.346380
3,62Z0101426517156,48.567394,30.232208
4,62Z013852333354Y,48.566059,30.230684


In [7]:
def make_nhits(h):
    """Build an N-HiTS model for a given forecast horizon h."""
    return NHITS(
        h=h,
        input_size=INPUT_SIZE,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
        # Multi-scale pooling: weekly (168h) -> daily (24h) -> hourly (1h)
        n_blocks=[3, 3, 3],             # was [1, 1, 1]
        mlp_units=[[512, 512], [512, 512], [512, 512]],
        n_pool_kernel_size=[168, 24, 1],
        n_freq_downsample=[168, 24, 1],
        batch_size=32,
        windows_batch_size=256,         # reduced from 1024 to avoid OOM
        step_size=24,                   # stride 24h -> ~24x fewer windows, much lower memory
        learning_rate=5e-4,             # was 1e-3
        max_steps=2000,                 # was 1000
        val_check_steps=50,
        early_stop_patience_steps=10,   # was 5
        scaler_type='robust',
        loss=SMAPE(),                   # was RMSE — aligns training with evaluation metric
    )

## Evaluation on Val and Test

Single fit on `train`; context is extended to `train+val` when predicting test.
No second fit needed — the model weights are reused with a longer context window.

In [8]:
# Fit once on train data; val_size sets aside the last HORIZON steps for early stopping
nf_eval = NeuralForecast(models=[make_nhits(HORIZON)], freq='h')
nf_eval.fit(df=train_nf, static_df=static_df, val_size=HORIZON)

Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ SMAPE         │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  127 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 127 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 M                                                                                                
Total estimated model params size (MB): 510                                                                        
Modules in train mode: 94                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

In [9]:
def build_futr_df(context_df, exog_source_df, h, exog_cols):
    """
    Build futr_df with exactly h steps per unique_id starting right after
    context_df ends. Merges exog features from exog_source_df; fills any
    gaps with forward/back fill so NeuralForecast never sees missing rows.
    """
    last_ts = context_df.groupby("unique_id")["ds"].max()
    frames = []
    for uid, last in last_ts.items():
        future_ds = pd.date_range(last + pd.Timedelta(hours=1), periods=h, freq="h")
        frames.append(pd.DataFrame({"unique_id": uid, "ds": future_ds}))
    futr_base = pd.concat(frames, ignore_index=True)
    futr = futr_base.merge(
        exog_source_df[["unique_id", "ds"] + exog_cols],
        on=["unique_id", "ds"], how="left",
    )
    futr[exog_cols] = futr.groupby("unique_id")[exog_cols].ffill().bfill()
    return futr

In [10]:
val_futr    = build_futr_df(train_nf, val_nf, HORIZON, FUTR_EXOG)
val_pred_df = nf_eval.predict(futr_df=val_futr)

val_merged  = val_nf[['unique_id', 'ds', 'y']].merge(val_pred_df, on=['unique_id', 'ds'])

print('Validation metrics')
print('SMAPE:', smape(val_merged['y'], val_merged['NHITS']))
print('RMSE :', rmse(val_merged['y'],  val_merged['NHITS']))
print('MAPE :', mape(val_merged['y'],  val_merged['NHITS']), '%')

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Validation metrics
SMAPE: 0.2411297363542331
RMSE : 10.330819059916328
MAPE : 29.982175362090153 %


In [11]:
# Predict test using the same trained model — just extend the context to train+val
train_val_nf = (
    pd.concat([train_nf, val_nf])
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

In [12]:
test_futr    = build_futr_df(train_val_nf, test_nf, HORIZON, FUTR_EXOG)
test_pred_df = nf_eval.predict(df=train_val_nf, static_df=static_df, futr_df=test_futr)

test_merged  = test_nf[['unique_id', 'ds', 'y']].merge(test_pred_df, on=['unique_id', 'ds'])

print('Test metrics')
print('SMAPE:', smape(test_merged['y'], test_merged['NHITS']))
print('RMSE :', rmse(test_merged['y'],  test_merged['NHITS']))
print('MAPE :', mape(test_merged['y'],  test_merged['NHITS']), '%')

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Test metrics
SMAPE: 0.23222702908499676
RMSE : 10.047466919164213
MAPE : 39.464000288520005 %


## 6-Month Prediction on `predict_X` (Recursive)

Retrain on **all** labelled data (train + val + test) with `h=HORIZON` (1 month), then
roll forward one month at a time for 6 steps.  
Each step feeds the previous predictions back as context for the next step.

In [13]:
predict_X_raw = load_and_prepare('data/no_y_col/with_weather_v2.parquet')
predict_nf    = prepare_nf(predict_X_raw, has_y=False)

# Compute how many hours per location we need to forecast
h_final = int(predict_nf['unique_id'].value_counts().min())
print(f'Prediction horizon (hours): {h_final}  (~{h_final/24:.1f} days)')

No Y_col: Sum of кВт
Prediction horizon (hours): 5089  (~212.0 days)


In [14]:
all_data_nf = (
    pd.concat([train_nf, val_nf, test_nf])
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

# Train with h=HORIZON (1 month); recursive steps will handle the full 6-month window
nf_final = NeuralForecast(models=[make_nhits(HORIZON)], freq='h')
nf_final.fit(df=all_data_nf, static_df=static_df, val_size=HORIZON)

Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ SMAPE         │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  127 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 127 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 M                                                                                                
Total estimated model params size (MB): 510                                                                        
Modules in train mode: 94                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

IndexError: index 410 is out of bounds for dimension 0 with size 410

In [ ]:
# --- Recursive 6-month forecast: roll forward one HORIZON at a time ---
predict_futr = predict_nf[['unique_id', 'ds'] + FUTR_EXOG]

context_nf   = all_data_nf.copy()
monthly_preds = []

max_rows = predict_futr.groupby('unique_id').size().max()
n_steps  = int(np.ceil(max_rows / HORIZON))
print(f'Total steps: {n_steps}  ({HORIZON}h each)')

for step in range(n_steps):
    lo, hi = step * HORIZON, (step + 1) * HORIZON

    step_futr = (
        predict_futr
        .groupby('unique_id', group_keys=False)
        .apply(lambda g: g.iloc[lo:hi])
        .reset_index(drop=True)
    )
    if step_futr.empty:
        break

    # Predict the next HORIZON hours using the current context
    step_pred = nf_final.predict(df=context_nf, futr_df=step_futr)
    monthly_preds.append(step_pred.copy())

    # Append predicted values back into context for the next step
    new_rows = step_futr.merge(
        step_pred.rename(columns={'NHITS': 'y'})[['unique_id', 'ds', 'y']],
        on=['unique_id', 'ds'],
    )
    context_nf = (
        pd.concat([context_nf, new_rows[context_nf.columns]])
        .sort_values(['unique_id', 'ds'])
        .reset_index(drop=True)
    )
    print(f'  Step {step + 1}/{n_steps} done — context rows: {len(context_nf)}')

# Combine all steps and restore original column names
final_preds = (
    pd.concat(monthly_preds)
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
    .rename(columns={'unique_id': ID_COL, 'ds': DS_COL, 'NHITS': Y_COL})
)
print(final_preds.shape)
final_preds.head(10)

In [ ]:
import os
os.makedirs('predictions', exist_ok=True)
final_preds.to_parquet('predictions/nhits_6month.parquet', index=False)
print('Saved to predictions/nhits_6month.parquet')